In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report)

from imblearn.over_sampling import SMOTE

from lightgbm import LGBMClassifier

In [2]:
df = pd.read_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\model_ready_dataset.csv")

df.head()

,HDF,OSF,PWF,TWF,rpm_torque_interaction,Rotational speed [rpm],load_stress,Torque [Nm],load_density,Tool wear [min],temperature_ratio,tool_wear_mean_10,temperature_difference,air_temp_mean_10,UDI,Machine failure
0,0,0,0,0,71177.0,1306,29.7025,54.5,0.545,50,1.034806,36.8,10.4,298.60,19,0
1,0,0,0,0,53040.0,1632,10.5625,32.5,0.325,55,1.034794,40.2,10.4,298.64,20,0
2,0,0,0,0,58712.5,1375,18.2329,42.7,0.427,58,1.034794,43.6,10.4,298.69,21,0
3,0,0,0,0,64960.0,1450,20.0704,44.8,0.448,63,1.035141,47.0,10.5,298.71,22,0
4,0,0,0,0,48536.7,1581,9.4249,30.7,0.307,65,1.034794,50.1,10.4,298.74,23,0


In [3]:
X = df.drop("Machine failure", axis=1)

y = df["Machine failure"]

In [4]:
X.columns = (
    X.columns
    .str.replace("[","",regex=False)
    .str.replace("]","",regex=False)
    .str.replace("{","",regex=False)
    .str.replace("}","",regex=False)
    .str.replace(":","",regex=False)
    .str.replace(",","",regex=False)
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
smote = SMOTE(random_state=42)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

y_train_balanced.value_counts()

Machine failure
0    7714
1    7714
Name: count, dtype: int64

In [6]:
model = LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=50, random_state=42)

In [7]:
model.fit(X_train_balanced, y_train_balanced)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 7714, number of negative: 7714
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000478 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 15428, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


LGBMClassifier(learning_rate=0.05, n_estimators=200, num_leaves=50,
               random_state=42)

In [8]:
y_pred = model.predict(X_test)

In [9]:
metrics = {
    
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred)
}

In [ ]:
metrics

{'Accuracy': 0.9964947421131698,
 'Precision': np.float64(0.9066666666666666),
 'Recall': np.float64(1.0),
 'F1 Score': np.float64(0.951048951048951)}

In [12]:
print("Classification Report:\n", classification_report(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      1929
           1       0.91      1.00      0.95        68

    accuracy                           1.00      1997
   macro avg       0.95      1.00      0.97      1997
weighted avg       1.00      1.00      1.00      1997



In [13]:
cm = confusion_matrix(y_test, y_pred)

cm

array([[1922,    7],
       [   0,   68]])

In [14]:
pd.DataFrame([metrics]).to_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\final_model_results.csv", index=False)

In [15]:
import joblib

joblib.dump(model, r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\models\final_lightgbm_model.pkl")

['E:\\AARAV\\Infotact-DS-ML\\Project-1-Predictive-Maintenance\\models\\final_lightgbm_model.pkl']